# CS781 capstone — MASTER notebook (all four architectures, Phases 3–7)

**How to use:** edit only the **CONTROL** cell (architecture, phase, technique, run mode, your hyperparameters), then Run All / commit.
Every phase section starts with `if PHASE == ...` so only the chosen phase runs.

**Who wrote what (for the LLM declaration):**
- *Your code, copied verbatim from your notebooks:* tokenizer (`a1_tokenizer.py`, `load_tokenizer`), the full Phase-1 code of arch 2 / 3 / 4 (one module each), your Phase-2 mask-aware arch-4 code, `load_encoder`, `load_encoder_a4`, `LoRALinear`, `LoRMALinear`, `inject`, `freeze_all_but`, `trainable_report`, and `backbone_states` (= the body of your Phase-2 `Tagger.forward`).
- *Plumbing written by Claude:* control cell layout, data download/loading/statistics, sentence packing, checkpoint finding, training/eval harness, macro-F1, submission writer, teacher-logit caching.
- *YOUR slots (marked `YOUR CODE`):* arch-1 classes (file not received yet), the Phase-3 model (pooling + document layer + head), PEFT techniques other than LoRA/LoRMA, the KD student and KD loss, and every hyperparameter.

## 0. CONTROL — the only cell you edit per run

In [ ]:
ARCH      = "arch1"     # "arch1" | "arch2" | "arch3" | "arch4"
PHASE     = 3           # 3 | 4 | 5 | 6 | 7
TECHNIQUE = "rslora"      # must be on this phase's list in the PDF (Phase 3: lora, qlora, adalora, rslora, lorma, prefix, prompt, ia3, kd)
RUN_MODE  = "dev"       # "dev": train on train, select on dev | "final": train on train+dev, predict test, write submission.zip
BIDIR     = False        # True = bidirectional (MLM checkpoints) | False = causal (arch1, or original LM checkpoints)
SEED      = 0

# checkpoint file names, searched anywhere under /kaggle/input
CKPTS = {
    "arch1": {"lm": "phase1_arch1_ckpt.pt"},   # your best arch-1 checkpoint (step 375, val 6.652); use the exact file name you attach
    "arch2": {"lm": "arch2e2_ckpt.pt", "mlm": "arch2_mlm_best.pt"},
    "arch3": {"lm": "arch3_ckpt.pt",   "mlm": "arch3_tapt.pt"},   # TAPT encoder (attach your arch3-tapt dataset),
    "arch4": {"lm": "arch4_ckpt.pt",   "mlm": "arch4_mlm_best.pt"},
}
CKPT_KIND = "mlm" if BIDIR else "lm"      # Phase 6 (generation) must use "lm"

# ---- YOUR hyperparameters: every None must be filled before a training run ----
HP = dict(
    R=8, ALPHA=8, TARGETS=("q_proj", "k_proj", "v_proj", "out_proj")   # arch-1 CausalSelfAttention uses these same names
   ,   # technique
    MAX_SENT_TOK=166, PACK_W=1024, SENT_SEP_EOS=True, MAX_TRAIN_WIN=4,     # Phase-3 input
    EPOCHS=30, LR_ENC=4e-4, LR_HEAD=1e-4, WARMUP_FRAC=0.1, WD=0.01,          # optimisation
    ACCUM_DOCS=1, CLASS_W_POWER=0.2,
    FINAL_EPOCHS=None,                       # RUN_MODE="final": fixed epochs (no dev split to select on)
    STAGE2_EPOCHS=60, STAGE2_LR=2e-5,      # 3.6 stage 2 (doc layer on full documents) -- your values
    FOCAL_GAMMA=2,                        # used only when LOSS = "focal"
    SHIFT_WEIGHT=0,                        # Stage-2 label-shift task weight (needs your shift_head + shift_loss)
    STUDENT_LAYERS=2, KD_T=3, KD_ALPHA=0.7,   # KD runs only
)
TEACHER_TAG = None   # KD runs: tag of the teacher run whose logits are attached, e.g. "arch3_lora"
CACHE_TEACHER_LOGITS = False   # True on the teacher's run: saves its logits on the training docs for KD
LOSS = "ce"          # "ce" = weighted cross-entropy | "focal" = your focal_loss (cell after 3.4)

def need(*keys):
    missing = [k for k in keys if HP.get(k) is None]
    assert not missing, f"fill these in the CONTROL cell first: {missing}"

RUN_TAG = f"p{PHASE}_{ARCH}_{TECHNIQUE}_{RUN_MODE}"
print("RUN:", RUN_TAG, "| checkpoint kind:", CKPT_KIND)

## 1. Setup

In [ ]:
import os, json, glob, math, time, random, zipfile, collections, re, bisect
import numpy as np, torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass, asdict

device = "cuda" if torch.cuda.is_available() else "cpu"
DEV_TYPE = "cuda" if device == "cuda" else "cpu"
EOS_ID, MASK_ID = 65354, 65355
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("device:", device)

def find_input(name):
    if name.startswith("/") and os.path.exists(name): return name
    hits = [os.path.join(r, n) for r, ds, fs in os.walk("/kaggle/input", followlinks=True)
            for n in fs + ds if n == name]
    assert hits, f"{name} not found under /kaggle/input -- attach the dataset/notebook output that holds it (restart the session if you attached it after starting)"
    return sorted(hits, key=os.path.isdir)[0]          # prefer a real file over a folder

In [ ]:
# official data, per phase (files as listed in the capstone PDF)
REPO = "Exploration-Lab/CS-781-Capstone"
PHASE_FILES = {
    3: ("Phase_3_RR",   ["Train/CL_train.json", "Train/CL_dev.json", "Train/IT_train.json", "Train/IT_dev.json",
                         "Train/label_vocab.json", "Test/CL_test.json", "Test/IT_test.json"]),
    4: ("Phase_4_CJPE", ["Train/train.json", "Train/dev.json", "Test/test.json"]),
    5: ("Phase_5_LSI",  ["Train/train.json", "Train/dev.json", "Test/test.json"]),
    6: ("Phase_6_Summ", ["Train/train.json", "Test/test.json"]),
    7: ("Phase_7_PCR",  ["Train/train.json", "Train/train_candidates.json", "Dev/dev.json",
                         "Dev/dev_candidates.json", "Test/test.json", "Test/test_candidates.json"]),
}

def fetch_phase(phase):
    """Uses attached copies if present (offline-safe), otherwise downloads with HF_TOKEN."""
    folder, files = PHASE_FILES[phase]
    paths = {}
    for f in files:
        local = glob.glob(f"/kaggle/input/**/{os.path.basename(f)}", recursive=True)
        if local and phase == 3:                  # phase-3 file names are unique; later phases share names (train.json ...)
            paths[f] = local[0]; continue
        from kaggle_secrets import UserSecretsClient
        from huggingface_hub import hf_hub_download
        tokn = UserSecretsClient().get_secret("HF_TOKEN")
        paths[f] = hf_hub_download(REPO, f"{folder}/{f}", repo_type="dataset", token=tokn)
    for f, p in paths.items():
        print(f"{f:32s} {os.path.getsize(p) // 1024:>8} KB  {p}")
    return paths

P = fetch_phase(PHASE)

## 2. Tokenizer (your Assignment-1 BPE, verbatim)

In [ ]:
%%writefile a1_tokenizer.py
"""Expose your Assignment-1 BPE implementation to Assignment 3.

Reuse your A-1 tokenizer code/class here. The important change is that it must be
RETRAINED on the `train_text` supplied by Assignment 3 rather than reusing a tokenizer
trained on the Assignment-1 dataset.
"""
import re
from collections import Counter

PAT=re.compile(r" ?[A-Za-z]+| ?\d+| ?[^\sA-Za-z\d]+|\s+")
class BPETokeniser:#defining the class
    def __init__(self):
        self.merges={}
        self.vocab={i: bytes([i]) for i in range(256)}
        self.cache={}

    @property
    def vocab_size(self):#vocab size
        return len(self.vocab)

    def _count_pairs(self,words,counts):#how actuall work we start counting how many words have same pair 
        pair_counts={}
        for word,count in zip(words,counts):
            for a,b in zip(word,word[1:]):
                pair_counts[(a,b)]=pair_counts.get((a,b),0)+count
        return pair_counts
    def _merge(self,ids,pair,new_id):
        out=[]
        i=0
        while i<len(ids):
            if i+1<len(ids) and ids[i]==pair[0] and ids[i+1]==pair[1]:
                out.append(new_id)
                i+=2
            else:
                out.append(ids[i])
                i+=1
        return out
    def train(self,text,vocab_size):
        chunks=PAT.findall(text)
        chunk_counts=Counter(chunks)
        words=[]
        counts=[]
        for chunk,n in chunk_counts.items():
            words.append(list(chunk.encode("utf-8")))
            counts.append(n)
        for r in range(vocab_size-256):
                pair_counts=self._count_pairs(words,counts)
                if not pair_counts:
                    break
                best = min(pair_counts, key=lambda p: (-pair_counts[p], p))
                new_id=256+r
                self.merges[best]=new_id
                self.vocab[new_id]=self.vocab[best[0]] + self.vocab[best[1]]
                words=[self._merge(w,best,new_id) for w in words]
    def _encode_chunk(self,chunk):
        if chunk in self.cache:
            return self.cache[chunk]
        ids=list(chunk.encode("utf-8"))
        while True:
            candidates=[p for p in zip(ids,ids[1:]) if p in self.merges]
            if not candidates:
                break
            best=min(candidates,key=lambda p:self.merges[p])
            ids=self._merge(ids,best,self.merges[best])
        self.cache[chunk]=ids
        return ids
    def encode(self,text):
        ids=[]
        for chunk in PAT.findall(text):
            ids.extend(self._encode_chunk(chunk))
        return ids
    def decode(self,ids):
        data=b"".join(self.vocab[i] for i in ids)
        return data.decode("utf-8",errors="replace")
             
         

def train_tokenizer(train_text: str, vocab_size: int):
    """Train your A-1 BPE tokenizer on `train_text` and return the trained tokenizer.

    Required returned interface:
      tokenizer.encode(text) -> sequence of integer ids
      tokenizer.decode(ids) -> string
      tokenizer.vocab_size OR tokenizer.get_vocab_size()

    Replace this stub with your Assignment-1 tokenizer implementation/training call.
    """
    tok=BPETokeniser()
    tok.train(train_text,vocab_size)
    return tok

In [ ]:
import re
from a1_tokenizer import BPETokeniser
mp = find_input("legal_bpe_merges.json")
raw = json.load(open(mp))
print(type(raw).__name__, len(raw))

In [ ]:
def load_tokenizer(raw):
    tok = BPETokeniser()
    if isinstance(raw, dict) and "merges" in raw:
        raw = raw["merges"]
    triples = []
    if isinstance(raw, dict):                                  # {"a,b": new_id}
        for k, v in raw.items():
            a, b = map(int, re.split(r"[,\s]+", k.strip("()[] ")))
            triples.append((a, b, int(v)))
    else:                                                      # list form
        for i, m in enumerate(raw):
            triples.append((int(m[0]), int(m[1]), int(m[2]) if len(m) == 3 else 256 + i))
    for a, b, new in sorted(triples, key=lambda t: t[2]):      # rebuild in learned order
        tok.merges[(a, b)] = new
        tok.vocab[new] = tok.vocab[a] + tok.vocab[b]
    return tok

tok = load_tokenizer(raw)
print("merges:", len(tok.merges), "| vocab:", tok.vocab_size)   # expect 65,098 | 65,354

## 3. The four architectures — each one complete, in its own module
Each cell below writes **one architecture's full Phase-1 code, verbatim from your Phase-1 notebook**, into its own file
(`arch1_model.py` … `arch4_model.py`) and imports it as `A1` … `A4`. Keeping them as separate modules means the four
sets of classes never overwrite each other (all of them define `GQAttention`, `RMSNorm`, …), and each architecture can be
shown on its own. A quick parameter count (on the `meta` device, so no memory is used) checks each one against Phase 1.

### 3.1 Architecture 1 — vanilla decoder (your A3 attention, sinusoidal positions, GELU FFN)
**YOUR CODE:** paste the model classes from `phase1-train (7).ipynb` into this cell, plus `load_encoder_a1(path) -> (encoder, cfg)` and `states_a1(encoder, idx) -> (B, T, d_model)` (hidden states after the final norm).

In [ ]:
%%writefile arch1_model.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

# ===== arch 1: your Phase-1 code, verbatim from phase1-train (7).ipynb (attention.py + model_arch.py) =====
#this is the same attention that i built in the A3 question 1


import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int, dropout: float = 0.0, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        
        self.q_proj = nn.Linear(d_model,d_model,bias=False)
        self.k_proj = nn.Linear(d_model,d_model,bias=False)
        self.v_proj = nn.Linear(d_model,d_model,bias=False)
        self.out_proj = nn.Linear(d_model,d_model,bias=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        
        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask",mask, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
       
        B, T, C = x.shape
        
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)
        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        scores = (q * self.attention_scale) @ k.transpose(-2, -1)
        mask = self.causal_mask[:T, :T]
        scores = scores.masked_fill(~mask, float("-inf"))
        weights = F.softmax(scores, dim=-1)
        weights = self.attn_dropout(weights)
        out = weights @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.resid_dropout(self.out_proj(out))
        return out


"""Decoder-only Transformer LM, built from scratch (no nn.Transformer*)."""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 512
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4          
    d_ff: int = 1024
    max_seq_len: int =1024
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   
    attention_scale: float = None      
    norm_first: bool = False            

def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    # TODO: implement PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
    #               PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    pos = torch.arange(max_len).unsqueeze(1).float()
    i2 = torch.arange(0, d_model, 2).float()
    div = torch.pow(10000.0, i2 / d_model)
    pe = torch.zeros(max_len, d_model)
    pe[:, 0::2] = torch.sin(pos / div)
    pe[:, 1::2] = torch.cos(pos / div)
    return pe
    


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        # TODO: two linear layers with a GELU in between, plus dropout
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """
    Pre-norm Transformer block: x + Attn(LN(x)), then x + FFN(LN(x)).
    OR
    Post-norm Transformer block: LN(x + Attn(x)), then LN(x + FFN(x)).
    """

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.max_seq_len, cfg.dropout,
                                        attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)
        self.norm_first = cfg.norm_first  

    def forward(self, x):
        # TODO: implement pre-norm when norm_first=True,
        # and post-norm when norm_first=False.
        if self.norm_first:
            x = x + self.attn(self.ln1(x))
            x = x + self.ffn(self.ln2(x))
        else:
            x = self.ln1(x + self.attn(x))
            x = self.ln2(x + self.ffn(x))
        return x
        


class DecoderLM(nn.Module):
    """The model class the autograder imports directly. Do not rename it."""

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)

        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)
        # (RoPE is applied inside attention instead of here — see Question 3)

        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying
        nn.init.normal_(self.tok_emb.weight, mean=0.0, std=cfg.d_model ** -0.5)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        """idx: (B, T) LongTensor of token ids -> logits (B, T, vocab_size)."""
        B, T = idx.shape
        # TODO: x = token embedding + positional encoding (if sinusoidal)
        # TODO: run x through every block, then ln_f, then self.head
        x = self.tok_emb(idx) * math.sqrt(self.cfg.d_model)
        
        if self.cfg.pos_emb_type == "sinusoidal":
            x = x + self.pos_emb[:T]
            
        for block in self.blocks:
            x = block(x)
            
        x = self.ln_f(x)
        return self.head(x * self.cfg.d_model ** -0.5)


# ---- loader + hidden states (plumbing around your classes) ----
def load_encoder_a1(path):
    """Your Phase-1 checkpoint ({"model", "cfg", ...}) -> (DecoderLM, LMConfig)."""
    ck = torch.load(path, map_location="cpu", weights_only=False)
    cfg = LMConfig(**ck["cfg"])
    enc = DecoderLM(cfg)
    enc.load_state_dict(ck["model"])
    print(path.split("/")[-1], "| step", ck.get("step"), "| best_val", round(ck.get("best_val") or float("nan"), 3),
          "| vocab", cfg.vocab_size, f"| {sum(p.numel() for p in enc.parameters()):,} params")
    del ck
    return enc, cfg

def states_a1(enc, idx):
    """= your DecoderLM.forward without the LM head: hidden states after ln_f (causal, mask is built in)."""
    T = idx.size(1)
    x = enc.tok_emb(idx) * math.sqrt(enc.cfg.d_model)
    if enc.cfg.pos_emb_type == "sinusoidal":
        x = x + enc.pos_emb[:T]
    for block in enc.blocks:
        x = block(x)
    return enc.ln_f(x)


### 3.2 Architecture 2 — LLaMA-style decoder (RoPE, RMSNorm, SwiGLU, GQA) — from `phase1-arch2-6.ipynb`

In [ ]:
%%writefile arch2_model.py
import math, torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

class RMSNorm(nn.Module):
    def __init__(self,d_model,eps=1e-6):
        super().__init__()
        self.eps=eps
        self.g=nn.Parameter(torch.ones(d_model))
    def forward(self,x):
        meanSquare=x.pow(2).mean(dim=-1,keepdim=True)
        x=x*torch.rsqrt(meanSquare+self.eps)
        return x*self.g

def rope_tables(max_seq_len,d_head,base=10000.0):
    assert d_head%2==0
    i=torch.arange(0,d_head//2).float()
    theta=base**(-2*i/d_head)
    m=torch.arange(max_seq_len).float()
    angles=torch.outer(m,theta)
    return torch.cos(angles),torch.sin(angles)

def apply_rope(x,cos,sin):
    T=x.size(2)
    cos,sin=cos[:T],sin[:T]
    x1=x[...,0::2]
    x2=x[...,1::2]
    r1=x1*cos-x2*sin
    r2=x1*sin+x2*cos
    out=torch.stack([r1,r2],dim=-1)
    return out.flatten(-2)

class GQAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, rope_base=10000.0):
        super().__init__()
        assert d_model%n_heads==0 and n_heads%n_kv_heads==0
        self.n_heads=n_heads
        self.n_kv_heads=n_kv_heads
        self.d_head=d_model // n_heads
        self.n_rep=n_heads // n_kv_heads
        self.scale=1/math.sqrt(self.d_head)
        self.q_proj=nn.Linear(d_model,d_model,bias=False)
        self.k_proj=nn.Linear(d_model,n_kv_heads*self.d_head,bias=False)
        self.v_proj=nn.Linear(d_model,n_kv_heads*self.d_head,bias=False)
        self.out_proj=nn.Linear(d_model,d_model,bias=False)
        self.attn_dropout=nn.Dropout(dropout)
        self.resid_dropout=nn.Dropout(dropout)
        mask=torch.tril(torch.ones(max_seq_len,max_seq_len,dtype=torch.bool))
        self.register_buffer("causal_mask",mask,persistent=False)
        cos,sin=rope_tables(max_seq_len,self.d_head, rope_base)
        self.register_buffer("cos",cos,persistent=False)
        self.register_buffer("sin",sin,persistent=False)
    def forward(self,x):
        B,T,C=x.shape
        q=self.q_proj(x).view(B,T,self.n_heads,self.d_head).transpose(1,2)
        k=self.k_proj(x).view(B,T,self.n_kv_heads,self.d_head).transpose(1,2)
        v=self.v_proj(x).view(B,T,self.n_kv_heads,self.d_head).transpose(1,2)
        q=apply_rope(q,self.cos,self.sin)
        k=apply_rope(k,self.cos,self.sin)
        k=k.repeat_interleave(self.n_rep,dim=1)
        v=v.repeat_interleave(self.n_rep,dim=1)
        scores=(q*self.scale)@k.transpose(-2,-1)
        scores=scores.masked_fill(~self.causal_mask[:T,:T],float("-inf"))
        weights=self.attn_dropout(F.softmax(scores,dim=-1))
        out=(weights@v).transpose(1,2).contiguous().view(B,T,C)
        return self.resid_dropout(self.out_proj(out))

class SwiGLU(nn.Module):
    def __init__(self,d_model,d_ff,dropout=0.0):
        super().__init__()
        self.w1=nn.Linear(d_model,d_ff,bias=False)
        self.w3=nn.Linear(d_model,d_ff,bias=False)
        self.w2=nn.Linear(d_ff,d_model,bias=False)
        self.dropout=nn.Dropout(dropout)
    def forward(self,x):
        gate=F.silu(self.w1(x))
        return self.dropout(self.w2(gate*self.w3(x)))

from dataclasses import dataclass

@dataclass
class LlamaConfig:
    vocab_size:int
    d_model:int
    n_layers:int
    n_heads:int
    n_kv_heads:int
    d_ff:int
    max_seq_len:int=1024
    dropout:float=0.0
    rope_base:float=10000.0

class LlamaBlock(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.norm1=RMSNorm(cfg.d_model)
        self.attn=GQAttention(cfg.d_model,cfg.n_heads,cfg.n_kv_heads,cfg.max_seq_len,cfg.dropout,cfg.rope_base)
        self.norm2=RMSNorm(cfg.d_model)
        self.ffn=SwiGLU(cfg.d_model,cfg.d_ff,cfg.dropout)
    def forward(self,x):
        x=x+self.attn(self.norm1(x))
        x=x+self.ffn(self.norm2(x))
        return x

class LlamaLM(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.cfg=cfg
        self.tok_emb=nn.Embedding(cfg.vocab_size,cfg.d_model)
        self.blocks=nn.ModuleList([LlamaBlock(cfg) for _ in range(cfg.n_layers)])
        self.norm_f=RMSNorm(cfg.d_model)
        self.head=nn.Linear(cfg.d_model,cfg.vocab_size,bias=False)
        self.head.weight=self.tok_emb.weight
        nn.init.normal_(self.tok_emb.weight,mean=0.0,std=0.02)
    def forward(self,idx):
        x=self.tok_emb(idx)
        for block in self.blocks:
            x=block(x)
        x=self.norm_f(x)
        return self.head(x)

### 3.3 Architecture 3 — prefix-LM on the arch-2 blocks (prefix mask, mask-aware attention) — from `phase1-arch3-240218.ipynb`

In [ ]:
%%writefile arch3_model.py
import math, torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

class RMSNorm(nn.Module):
    def __init__(self,d_model,eps=1e-6):
        super().__init__()
        self.eps=eps
        self.g=nn.Parameter(torch.ones(d_model))
    def forward(self,x):
        meanSquare=x.pow(2).mean(dim=-1,keepdim=True)
        x=x*torch.rsqrt(meanSquare+self.eps)
        return x*self.g

def rope_tables(max_seq_len,d_head,base=10000.0):
    assert d_head%2==0
    i=torch.arange(0,d_head//2).float()
    theta=base**(-2*i/d_head)
    m=torch.arange(max_seq_len).float()
    angles=torch.outer(m,theta)
    return torch.cos(angles),torch.sin(angles)

def apply_rope(x,cos,sin):
    T=x.size(2)
    cos,sin=cos[:T],sin[:T]
    x1=x[...,0::2]
    x2=x[...,1::2]
    r1=x1*cos-x2*sin
    r2=x1*sin+x2*cos
    out=torch.stack([r1,r2],dim=-1)
    return out.flatten(-2)

def prefix_mask(T,prefix_lens,device=None):
    causal=torch.tril(torch.ones(T,T,dtype=torch.bool,device=device))
    pos=torch.arange(T,device=device)
    in_prefix=pos[None,:]<prefix_lens[:,None]
    both=in_prefix[:,:,None] & in_prefix[:,None,:]
    return (causal[None] | both)[:, None]

class GQAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, rope_base=10000.0):
        super().__init__()
        assert d_model%n_heads==0 and n_heads%n_kv_heads==0
        self.n_heads=n_heads
        self.n_kv_heads=n_kv_heads
        self.d_head=d_model // n_heads
        self.n_rep=n_heads // n_kv_heads
        self.scale=1/math.sqrt(self.d_head)
        self.q_proj=nn.Linear(d_model,d_model,bias=False)
        self.k_proj=nn.Linear(d_model,n_kv_heads*self.d_head,bias=False)
        self.v_proj=nn.Linear(d_model,n_kv_heads*self.d_head,bias=False)
        self.out_proj=nn.Linear(d_model,d_model,bias=False)
        self.attn_dropout=nn.Dropout(dropout)
        self.resid_dropout=nn.Dropout(dropout)
        mask=torch.tril(torch.ones(max_seq_len,max_seq_len,dtype=torch.bool))
        self.register_buffer("causal_mask",mask,persistent=False)
        cos,sin=rope_tables(max_seq_len,self.d_head, rope_base)
        self.register_buffer("cos",cos,persistent=False)
        self.register_buffer("sin",sin,persistent=False)
    def forward(self,x,mask=None):
        B,T,C=x.shape
        q=self.q_proj(x).view(B,T,self.n_heads,self.d_head).transpose(1,2)
        k=self.k_proj(x).view(B,T,self.n_kv_heads,self.d_head).transpose(1,2)
        v=self.v_proj(x).view(B,T,self.n_kv_heads,self.d_head).transpose(1,2)
        q=apply_rope(q,self.cos,self.sin)
        k=apply_rope(k,self.cos,self.sin)
        k=k.repeat_interleave(self.n_rep,dim=1)
        v=v.repeat_interleave(self.n_rep,dim=1)
        scores=(q*self.scale)@k.transpose(-2,-1)
        if mask is None:
            mask=self.causal_mask[:T,:T]
        scores=scores.masked_fill(~mask,float("-inf"))
        
        weights=self.attn_dropout(F.softmax(scores,dim=-1))
        out=(weights@v).transpose(1,2).contiguous().view(B,T,C)
        return self.resid_dropout(self.out_proj(out))

class SwiGLU(nn.Module):
    def __init__(self,d_model,d_ff,dropout=0.0):
        super().__init__()
        self.w1=nn.Linear(d_model,d_ff,bias=False)
        self.w3=nn.Linear(d_model,d_ff,bias=False)
        self.w2=nn.Linear(d_ff,d_model,bias=False)
        self.dropout=nn.Dropout(dropout)
    def forward(self,x):
        gate=F.silu(self.w1(x))
        return self.dropout(self.w2(gate*self.w3(x)))

from dataclasses import dataclass

@dataclass
class LlamaConfig:
    vocab_size:int
    d_model:int
    n_layers:int
    n_heads:int
    n_kv_heads:int
    d_ff:int
    max_seq_len:int=1024
    dropout:float=0.0
    rope_base:float=10000.0

class LlamaBlock(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.norm1=RMSNorm(cfg.d_model)
        self.attn=GQAttention(cfg.d_model,cfg.n_heads,cfg.n_kv_heads,cfg.max_seq_len,cfg.dropout,cfg.rope_base)
        self.norm2=RMSNorm(cfg.d_model)
        self.ffn=SwiGLU(cfg.d_model,cfg.d_ff,cfg.dropout)
    def forward(self,x,mask=None):
        x=x+self.attn(self.norm1(x),mask)
        x=x+self.ffn(self.norm2(x))
        return x

class PrefixLM(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.cfg=cfg
        self.tok_emb=nn.Embedding(cfg.vocab_size,cfg.d_model)
        self.blocks=nn.ModuleList([LlamaBlock(cfg) for _ in range(cfg.n_layers)])
        self.norm_f=RMSNorm(cfg.d_model)
        self.head=nn.Linear(cfg.d_model,cfg.vocab_size,bias=False)
        self.head.weight=self.tok_emb.weight
        nn.init.normal_(self.tok_emb.weight,mean=0.0,std=0.02)
    def forward(self,idx,prefix_lens=None):
        if prefix_lens is not None:
            mask=prefix_mask(idx.size(1),prefix_lens,idx.device)
        else:
            mask=None
        x=self.tok_emb(idx)
        for block in self.blocks: x=block(x,mask)
        x=self.norm_f(x)
        return self.head(x)

### 3.4 Architecture 4 — dense + MoE hybrid (4 experts, top-2, on odd layers) — from `phase1-arch4.ipynb`

In [ ]:
%%writefile arch4_model.py
import math, torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

class RMSNorm(nn.Module):
    def __init__(self,d_model,eps=1e-6):
        super().__init__()
        self.eps=eps
        self.g=nn.Parameter(torch.ones(d_model))
    def forward(self,x):
        meanSquare=x.pow(2).mean(dim=-1,keepdim=True)
        x=x*torch.rsqrt(meanSquare+self.eps)
        return x*self.g

def rope_tables(max_seq_len,d_head,base=10000.0):
    assert d_head%2==0
    i=torch.arange(0,d_head//2).float()
    theta=base**(-2*i/d_head)
    m=torch.arange(max_seq_len).float()
    angles=torch.outer(m,theta)
    return torch.cos(angles),torch.sin(angles)

def apply_rope(x,cos,sin):
    T=x.size(2)
    cos,sin=cos[:T],sin[:T]
    x1=x[...,0::2]
    x2=x[...,1::2]
    r1=x1*cos-x2*sin
    r2=x1*sin+x2*cos
    out=torch.stack([r1,r2],dim=-1)
    return out.flatten(-2)

class GQAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, rope_base=10000.0):
        super().__init__()
        assert d_model%n_heads==0 and n_heads%n_kv_heads==0
        self.n_heads=n_heads
        self.n_kv_heads=n_kv_heads
        self.d_head=d_model // n_heads
        self.n_rep=n_heads // n_kv_heads
        self.scale=1/math.sqrt(self.d_head)
        self.q_proj=nn.Linear(d_model,d_model,bias=False)
        self.k_proj=nn.Linear(d_model,n_kv_heads*self.d_head,bias=False)
        self.v_proj=nn.Linear(d_model,n_kv_heads*self.d_head,bias=False)
        self.out_proj=nn.Linear(d_model,d_model,bias=False)
        self.attn_dropout=nn.Dropout(dropout)
        self.resid_dropout=nn.Dropout(dropout)
        mask=torch.tril(torch.ones(max_seq_len,max_seq_len,dtype=torch.bool))
        self.register_buffer("causal_mask",mask,persistent=False)
        cos,sin=rope_tables(max_seq_len,self.d_head, rope_base)
        self.register_buffer("cos",cos,persistent=False)
        self.register_buffer("sin",sin,persistent=False)
    def forward(self,x):
        B,T,C=x.shape
        q=self.q_proj(x).view(B,T,self.n_heads,self.d_head).transpose(1,2)
        k=self.k_proj(x).view(B,T,self.n_kv_heads,self.d_head).transpose(1,2)
        v=self.v_proj(x).view(B,T,self.n_kv_heads,self.d_head).transpose(1,2)
        q=apply_rope(q,self.cos,self.sin)
        k=apply_rope(k,self.cos,self.sin)
        k=k.repeat_interleave(self.n_rep,dim=1)
        v=v.repeat_interleave(self.n_rep,dim=1)
        scores=(q*self.scale)@k.transpose(-2,-1)
        scores=scores.masked_fill(~self.causal_mask[:T,:T],float("-inf"))
        weights=self.attn_dropout(F.softmax(scores,dim=-1))
        out=(weights@v).transpose(1,2).contiguous().view(B,T,C)
        return self.resid_dropout(self.out_proj(out))

class SwiGLU(nn.Module):
    def __init__(self,d_model,d_ff,dropout=0.0):
        super().__init__()
        self.w1=nn.Linear(d_model,d_ff,bias=False)
        self.w3=nn.Linear(d_model,d_ff,bias=False)
        self.w2=nn.Linear(d_ff,d_model,bias=False)
        self.dropout=nn.Dropout(dropout)
    def forward(self,x):
        gate=F.silu(self.w1(x))
        return self.dropout(self.w2(gate*self.w3(x)))

class MoEFFN(nn.Module):
    def __init__(self,d_model,d_ff,n_experts,top_k,dropout=0.0):
        super().__init__()
        self.n_experts=n_experts
        self.top_k=top_k
        self.router=nn.Linear(d_model,n_experts,bias=False)
        self.experts=nn.ModuleList([SwiGLU(d_model,d_ff,dropout) for _ in range(n_experts)])
        self.aux_loss=torch.tensor(0.0)
    def forward(self,x):
        B,T,d=x.shape
        flat=x.reshape(-1,d)
        with torch.autocast("cuda",enabled=False):
            logits=self.router(flat.float())
        probs=F.softmax(logits,dim=-1)
        top_p,top_i=probs.topk(self.top_k,dim=-1)
        top_p=top_p/top_p.sum(-1,keepdim=True)
        out=torch.zeros_like(flat)
        for e,expert in enumerate(self.experts):
            tok,slot=(top_i==e).nonzero(as_tuple=True)
            if tok.numel()==0: continue
            y=expert(flat[tok])
            out.index_add_(0,tok,(y*top_p[tok,slot,None]).to(out.dtype))
        f=torch.bincount(top_i.flatten(),minlength=self.n_experts).float()/top_i.numel()
        P=probs.mean(0)
        self.aux_loss=self.n_experts*(f*P).sum()
        return out.view(B,T,d)

@dataclass
class HybridConfig:
    vocab_size:int
    d_model:int
    n_layers:int
    n_heads:int
    n_kv_heads:int
    d_ff:int
    max_seq_len:int=1024
    dropout:float=0.0
    rope_base:float=10000.0
    n_experts:int=4
    top_k:int=2
    d_ff_expert:int=1024

class HybridBlock(nn.Module):
    def __init__(self,cfg,layer_id):
        super().__init__()
        self.norm1=RMSNorm(cfg.d_model)
        self.attn=GQAttention(cfg.d_model,cfg.n_heads,cfg.n_kv_heads,cfg.max_seq_len,cfg.dropout,cfg.rope_base)
        self.norm2=RMSNorm(cfg.d_model)
        self.is_moe=(layer_id%2==1)
        if self.is_moe:
            self.ffn=MoEFFN(cfg.d_model,cfg.d_ff_expert,cfg.n_experts,cfg.top_k,cfg.dropout)
        else:
            self.ffn=SwiGLU(cfg.d_model,cfg.d_ff,cfg.dropout)
    def forward(self,x):
        x=x+self.attn(self.norm1(x))
        x=x+self.ffn(self.norm2(x))
        return x

class HybridLM(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.cfg=cfg
        self.tok_emb=nn.Embedding(cfg.vocab_size,cfg.d_model)
        self.blocks=nn.ModuleList([HybridBlock(cfg,i) for i in range(cfg.n_layers)])
        self.norm_f=RMSNorm(cfg.d_model)
        self.head=nn.Linear(cfg.d_model,cfg.vocab_size,bias=False)
        self.head.weight=self.tok_emb.weight
        nn.init.normal_(self.tok_emb.weight,mean=0.0,std=0.02)
    def forward(self,idx):
        x=self.tok_emb(idx)
        for block in self.blocks:
            x=block(x)
        x=self.norm_f(x)
        return self.head(x)
    @property
    def aux_loss(self):
        moe=[b.ffn.aux_loss for b in self.blocks if b.is_moe]
        return sum(moe)/len(moe)

In [ ]:
import importlib, arch1_model, arch2_model, arch3_model, arch4_model
for _m in (arch1_model, arch2_model, arch3_model, arch4_model): importlib.reload(_m)
A1, A2, A3, A4 = arch1_model, arch2_model, arch3_model, arch4_model

with torch.device("meta"):          # builds the Phase-1 configs without allocating memory
    n2 = sum(p.numel() for p in A2.LlamaLM(A2.LlamaConfig(65355, 1024, 10, 16, 8, 2816)).parameters())
    n3 = sum(p.numel() for p in A3.PrefixLM(A3.LlamaConfig(65355, 1024, 10, 16, 8, 2816)).parameters())
    n4 = sum(p.numel() for p in A4.HybridLM(A4.HybridConfig(65355, 768, 16, 12, 6, 2048)).parameters())
print(f"arch2 {n2:,} (Phase 1: 184,909,824) | arch3 {n3:,} (184,909,824) | arch4 {n4:,} (191,800,320)")

### 3.5 Fine-tuning versions (the `mask` argument you added in Phase 2)
Arch 2's and arch 4's Phase-1 attention is causal-only. In Phase 2 you ran them through the **mask-aware** blocks
(arch 3's `GQAttention`, and `HybridBlock.forward(x, mask)`), which load the same weights: the parameter names are identical,
and with no mask they compute exactly the causal Phase-1 model. The cell below is your Phase-2 arch-4 code, verbatim.

In [ ]:
%%writefile arch4_ft.py
import math, torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

from arch3_model import RMSNorm, rope_tables, apply_rope, prefix_mask, GQAttention, SwiGLU

# ===== arch 4: dense + MoE hybrid (classes copied from my Phase 1 notebook phase1-arch4) =====
class MoEFFN(nn.Module):
    def __init__(self,d_model,d_ff,n_experts,top_k,dropout=0.0):
        super().__init__()
        self.n_experts=n_experts
        self.top_k=top_k
        self.router=nn.Linear(d_model,n_experts,bias=False)
        self.experts=nn.ModuleList([SwiGLU(d_model,d_ff,dropout) for _ in range(n_experts)])
        self.aux_loss=torch.tensor(0.0)
    def forward(self,x):
        B,T,d=x.shape
        flat=x.reshape(-1,d)
        with torch.autocast("cuda",enabled=False):
            logits=self.router(flat.float())
        probs=F.softmax(logits,dim=-1)
        top_p,top_i=probs.topk(self.top_k,dim=-1)
        top_p=top_p/top_p.sum(-1,keepdim=True)
        out=torch.zeros_like(flat)
        for e,expert in enumerate(self.experts):
            tok,slot=(top_i==e).nonzero(as_tuple=True)
            if tok.numel()==0: continue
            y=expert(flat[tok])
            out.index_add_(0,tok,(y*top_p[tok,slot,None]).to(out.dtype))
        f=torch.bincount(top_i.flatten(),minlength=self.n_experts).float()/top_i.numel()
        P_=probs.mean(0)
        self.aux_loss=self.n_experts*(f*P_).sum()
        return out.view(B,T,d)

@dataclass
class HybridConfig:
    vocab_size:int
    d_model:int
    n_layers:int
    n_heads:int
    n_kv_heads:int
    d_ff:int
    max_seq_len:int=1024
    dropout:float=0.0
    rope_base:float=10000.0
    n_experts:int=4
    top_k:int=2
    d_ff_expert:int=1024

class HybridBlock(nn.Module):
    def __init__(self,cfg,layer_id):
        super().__init__()
        self.norm1=RMSNorm(cfg.d_model)
        self.attn=GQAttention(cfg.d_model,cfg.n_heads,cfg.n_kv_heads,cfg.max_seq_len,cfg.dropout,cfg.rope_base)
        self.norm2=RMSNorm(cfg.d_model)
        self.is_moe=(layer_id%2==1)
        if self.is_moe:
            self.ffn=MoEFFN(cfg.d_model,cfg.d_ff_expert,cfg.n_experts,cfg.top_k,cfg.dropout)
        else:
            self.ffn=SwiGLU(cfg.d_model,cfg.d_ff,cfg.dropout)
    def forward(self,x,mask=None):            # mask arg added so the Phase-2 Tagger can call block(x, mask)
        x=x+self.attn(self.norm1(x),mask)
        x=x+self.ffn(self.norm2(x))
        return x

class HybridLM(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.cfg=cfg
        self.tok_emb=nn.Embedding(cfg.vocab_size,cfg.d_model)
        self.blocks=nn.ModuleList([HybridBlock(cfg,i) for i in range(cfg.n_layers)])
        self.norm_f=RMSNorm(cfg.d_model)
        self.head=nn.Linear(cfg.d_model,cfg.vocab_size,bias=False)
        self.head.weight=self.tok_emb.weight
        nn.init.normal_(self.tok_emb.weight,mean=0.0,std=0.02)
    def forward(self,idx):
        x=self.tok_emb(idx)
        for block in self.blocks:
            x=block(x)
        return self.head(self.norm_f(x))

def load_encoder_a4(path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    cfg = HybridConfig(**ck["cfg"])
    enc = HybridLM(cfg); enc.load_state_dict(ck["model"])
    print(path.split("/")[-1], "| step", ck.get("step"), "| best_val", round(ck.get("best_val") or float("nan"), 3),
          "| vocab", cfg.vocab_size, f"| {sum(p.numel() for p in enc.parameters()):,} params",
          "| MoE layers", sum(b.is_moe for b in enc.blocks), "of", len(enc.blocks))
    del ck
    return enc, cfg

## 4. MLM adaptation — kept aside
MLM training runs in its own notebooks (`arch3-mlm`, and the `arch2-mlm` copy of it), not here.
Their outputs (`arch2_mlm_best.pt`, `arch3_mlm_best.pt`, later `arch4_mlm_best.pt`) are plain checkpoints in the usual
`{"model", "cfg"}` format, so section 5 loads them exactly like the Phase-1 checkpoints.

## 5. Backbone loader

In [ ]:
import arch4_ft; importlib.reload(arch4_ft)
from arch3_model import LlamaConfig, PrefixLM, prefix_mask      # used by your load_encoder below
from arch4_ft import load_encoder_a4

def load_encoder(path):
    ck = torch.load(path, map_location="cpu")
    cfg = LlamaConfig(**ck["cfg"])
    enc = PrefixLM(cfg); enc.load_state_dict(ck["model"])
    print(path.split("/")[-1], "| step", ck.get("step"), "| best_val", round(ck.get("best_val") or float("nan"), 3),
          "| vocab", cfg.vocab_size, f"| {sum(p.numel() for p in enc.parameters()):,} params")
    del ck
    return enc, cfg

def backbone_states(enc, idx, prefix_lens=None):
    """Your Phase-2 Tagger.forward without the tag head: hidden states after norm_f.
    prefix_lens = window lengths -> fully bidirectional over the real tokens; None -> causal."""
    mask = prefix_mask(idx.size(1), prefix_lens, idx.device) if prefix_lens is not None else None
    x = enc.tok_emb(idx)
    for block in enc.blocks:
        x = block(x, mask)
    return enc.norm_f(x)

def load_backbone(arch=None, kind=None):
    """arch2 / arch3 -> PrefixLM (arch-3 module), arch4 -> mask-aware HybridLM (arch4_ft), arch1 -> your arch1_model."""
    arch, kind = arch or ARCH, kind or CKPT_KIND
    assert kind in CKPTS[arch], f"{arch} has no '{kind}' checkpoint listed in CKPTS"
    path = find_input(CKPTS[arch][kind])
    if arch in ("arch2", "arch3"):
        enc, cfg = load_encoder(path)
    elif arch == "arch4":
        enc, cfg = load_encoder_a4(path)
    else:
        assert hasattr(A1, "load_encoder_a1"), "fill section 3.1 (arch 1) first"
        enc, cfg = A1.load_encoder_a1(path)
    return enc, cfg.d_model

def encoder_states(enc, win_ids, win_lens):
    """What your Phase-3 model calls: (n_win, L) ids + (n_win,) lengths -> (n_win, L, d_model)."""
    if ARCH == "arch1":
        return A1.states_a1(enc, win_ids)
    return backbone_states(enc, win_ids, win_lens if BIDIR else None)

## 6. Fine-tuning techniques
Your LoRA / LoRMA and helpers (verbatim), then `apply_technique`, then the slot for the techniques not written yet.

In [ ]:
# ===== PEFT plumbing: freeze the backbone, report what is trainable =====
def freeze_all_but(model, keep):
    """keep(name, param) -> True if this parameter should train."""
    for n, p in model.named_parameters():
        p.requires_grad = bool(keep(n, p))

def trainable_report(model, tag=""):
    seen = {id(p): p for p in model.parameters()}.values()
    tot = sum(p.numel() for p in seen); tr = sum(p.numel() for p in seen if p.requires_grad)
    print(f"{tag} trainable {tr:,} / {tot:,} params ({tr / tot:.3%})")

In [ ]:
# ===== LoRA layer (my implementation): W' = W + (alpha/r) B A, low-rank term on the INPUT x =====
import torch
import torch.nn as nn
import math
class LoRALinear(nn.Module):
    def __init__(self,base:nn.Linear,r:int,alpha:float):
        super().__init__()
        self.b=base
        for p in self.b.parameters():p.requires_grad=False
        i,o=base.in_features,base.out_features
        self.lora_A=nn.Parameter(torch.empty(r,i))
        self.lora_B=nn.Parameter(torch.zeros(o,r))
        nn.init.kaiming_uniform_(self.lora_A,a=math.sqrt(5))
        self.s=alpha/r
    def forward(self,x):
        return self.b(x)+self.s*(x@self.lora_A.T@self.lora_B.T)

In [ ]:
# ===== LoRMA layer (my implementation): W' = (I + (alpha/r) B A) W, applied to the frozen Linear's output =====
import torch
import torch.nn as nn
import math

class LoRMALinear(nn.Module):
    def __init__(self, base: nn.Linear, r: int, alpha: float):
        super().__init__()
        self.base = base

        for param in self.base.parameters():
            param.requires_grad = False

        d_out = base.out_features

        self.lorma_A = nn.Parameter(torch.empty(r, d_out))
        nn.init.kaiming_uniform_(self.lorma_A, a=math.sqrt(5))

        self.lorma_B = nn.Parameter(torch.zeros(d_out, r))

        self.scale = alpha / r

    def forward(self, x):
        h = self.base(x)
        return h + self.scale * (h @ self.lorma_A.T @ self.lorma_B.T)

In [ ]:
def inject(model, wrap_fn, targets):
    """Replace every nn.Linear child whose attribute name is in `targets` with wrap_fn(linear)."""
    n = 0
    for _, mod in list(model.named_modules()):
        for t in targets:
            child = getattr(mod, t, None)
            if isinstance(child, nn.Linear):
                setattr(mod, t, wrap_fn(child)); n += 1
    return n

In [ ]:
def apply_technique(model, technique=None):
    """Wraps model.encoder for the chosen technique and freezes accordingly.
    Rule used for every PEFT method: the technique's own parameters + everything outside model.encoder
    (your pooling / document layer / head) train; the rest of the encoder is frozen."""
    t = technique or TECHNIQUE
    if t == "full":
        for p in model.parameters(): p.requires_grad = True
    elif t == "lora":
        need("R", "ALPHA")
        n = inject(model.encoder, lambda lin: LoRALinear(lin, HP["R"], HP["ALPHA"]), HP["TARGETS"])
        freeze_all_but(model, lambda n_, p: ("lora" in n_) or not n_.startswith("encoder."))
        print("LoRA wrapped", n, "Linears")
    elif t == "lorma":
        need("R", "ALPHA")
        n = inject(model.encoder, lambda lin: LoRMALinear(lin, HP["R"], HP["ALPHA"]), HP["TARGETS"])
        freeze_all_but(model, lambda n_, p: ("lorma" in n_) or not n_.startswith("encoder."))
        print("LoRMA wrapped", n, "Linears")
    elif t in EXTRA_TECHNIQUES:
        EXTRA_TECHNIQUES[t](model)
    else:
        raise NotImplementedError(f"technique '{t}' not written yet -- add it to EXTRA_TECHNIQUES in the next cell")
    trainable_report(model, f"{ARCH}+{t}")
    return model

In [ ]:
def apply_rslora(model):
    need("R", "ALPHA")
    inject(model.encoder, lambda lin: LoRALinear(lin, HP["R"], HP["ALPHA"]), HP["TARGETS"])
    
    count = 0
    scale = HP["ALPHA"] / (HP["R"] ** 0.5)
    
    for module in model.encoder.modules():
        if isinstance(module, LoRALinear):
            module.s = scale
            count += 1
            
    freeze_all_but(model, lambda n_, p: ("lora" in n_) or not n_.startswith("encoder."))
    print("rsLoRA wrapped", count, "Linears | scale alpha/sqrt(r) =", round(scale, 4))

EXTRA_TECHNIQUES = {
    "rslora": apply_rslora,
}

# PHASE 3 — Rhetorical Role prediction

### 3.1 Data + statistics

In [ ]:
if PHASE == 3:
    def to_docs(obj):
        if isinstance(obj, list):                                            # [ {id, text, labels}, ... ]
            return obj
        vals = list(obj.values())
        if all(isinstance(v, dict) for v in vals):                           # {doc_id: {text, labels}}
            return [dict(id=k, **v) for k, v in obj.items()]
        if "text" in obj and isinstance(obj["text"], list) and obj["text"] and isinstance(obj["text"][0], list):
            n = len(obj["text"])                                             # columnar {id:[...], text:[[...]], labels:[[...]]}
            docs = [{k: obj[k][i] for k in obj if isinstance(obj[k], list) and len(obj[k]) == n} for i in range(n)]
            for i, d in enumerate(docs): d.setdefault("id", str(i))
            return docs
        for v in vals:
            if isinstance(v, list) and v and isinstance(v[0], dict):         # {"data": [ {...}, ... ]}
                return v
        if all(isinstance(v, list) and (not v or isinstance(v[0], str)) for v in vals):
            return [{"id": k, "text": v} for k, v in obj.items()]            # {doc_id: [sentences]}
        raise ValueError(f"unknown layout: top keys {list(obj)[:5]}, first value type {type(vals[0])}")

    vocab = json.load(open(P["Train/label_vocab.json"]))
    if isinstance(vocab, dict) and all(str(k).isdigit() for k in vocab):          # {"0": "Fact", ...}
        LABELS = [vocab[k] for k in sorted(vocab, key=int)]
    elif isinstance(vocab, dict):                                                  # {"Fact": 0, ...}
        LABELS = sorted(vocab, key=lambda k: vocab[k]) if all(isinstance(v, int) for v in vocab.values()) else list(vocab)
    else:
        LABELS = list(vocab)
    L2I = {l: i for i, l in enumerate(LABELS)}
    print("labels:", LABELS)

    RAW = {}
    for dom in ("CL", "IT"):
        for split in ("train", "dev", "test"):
            key = f"{'Test' if split == 'test' else 'Train'}/{dom}_{split}.json"
            obj = json.load(open(P[key]))
            try:
                RAW[(dom, split)] = to_docs(obj)
            except Exception as e:
                print(f"!! {key}: top-level type {type(obj).__name__}, "
                      f"first keys {list(obj)[:3] if isinstance(obj, dict) else '-'}, "
                      f"first item {str(obj[0] if isinstance(obj, list) else next(iter(obj.values())))[:200]}")
                raise
            d0 = RAW[(dom, split)][0]
            print(f"{dom}_{split:5s}: loaded {len(RAW[(dom, split)])} docs | keys {list(d0.keys())}")

    ids_cl = {d["id"] for s in ("train", "dev", "test") for d in RAW[("CL", s)]}
    ids_it = {d["id"] for s in ("train", "dev", "test") for d in RAW[("IT", s)]}
    test_ids = [d["id"] for dom in ("CL", "IT") for d in RAW[(dom, "test")]]
    assert len(test_ids) == len(set(test_ids)), "duplicate test ids across CL/IT -- the output dict would collide"
    print("CL/IT id overlap:", len(ids_cl & ids_it))

    def pct(xs, q):
        xs = sorted(xs); return xs[min(len(xs) - 1, int(q * len(xs)))]
    for (dom, split), docs in RAW.items():
        ns = [len(d["text"]) for d in docs]
        print(f"{dom}_{split:5s}: {len(docs):4d} docs | {sum(ns):6d} sentences | sent/doc p50 {pct(ns, .5)} max {max(ns)}")
        if "labels" in docs[0]:
            assert all(len(d["labels"]) == len(d["text"]) for d in docs), "label/sentence length mismatch"
            assert not ({l for d in docs for l in d["labels"]} - set(LABELS)), "unknown label in data"
    for dom in ("CL", "IT"):
        c = collections.Counter(l for s in ("train", "dev") for d in RAW[(dom, s)] for l in d["labels"])
        tot = sum(c.values())
        print(f"\n{dom} train+dev label share:", {l: f"{100 * c[l] / tot:.1f}%" for l in LABELS})

In [ ]:
def pct(xs, q): xs = sorted(xs); return xs[min(len(xs) - 1, int(q * len(xs)))]
for split in ("train", "dev", "test"):
    sl = [len(tok.encode(" " + s.strip())) for dom in ("CL", "IT") for d in RAW[(dom, split)] for s in d["text"]]
    dl = [sum(len(tok.encode(" " + s.strip())) for s in d["text"]) for dom in ("CL", "IT") for d in RAW[(dom, split)]]
    print(f"{split:5s} tokens/sentence p50 {pct(sl,.5)} p90 {pct(sl,.9)} p95 {pct(sl,.95)} p99 {pct(sl,.99)} max {max(sl)} "
          f"| tokens/doc p50 {pct(dl,.5)} p90 {pct(dl,.9)} max {max(dl)}")

### 3.1b TAPT (optional): continue MLM on the Phase-3 text
Set `RUN_TAPT = False` and the `TAPT_*` values; then 3.5 fine-tunes the TAPT encoder in the same session.

In [ ]:
# ===== 3.1b TAPT: task-adaptive MLM on Phase-3 text (Gururangan et al., 2020) — plumbing around YOUR MLM recipe =====
# Continues MLM of the ARCH encoder (its "mlm" checkpoint) on the Phase-3 sentences (train + dev + test, text only,
# no labels — confirmed with the TA). Masking = your mlm_mask from arch3-mlm (verbatim). Saves {ARCH}_tapt.pt in the
# usual {"model","cfg"} format and points CKPTS[ARCH]["mlm"] at it, so 3.5 in THIS session fine-tunes the TAPT encoder.
RUN_TAPT = False
TAPT_STEPS, TAPT_PEAK_LR, TAPT_MIN_LR, TAPT_WARMUP = None, None, None, None     # YOUR values
TAPT_MICRO_B, TAPT_ACCUM, TAPT_T, TAPT_MASK_P = 4, 122, 1024, 0.15               # your arch3-mlm values (change if you like)

if PHASE == 3 and RUN_TAPT:
    assert None not in (TAPT_STEPS, TAPT_PEAK_LR, TAPT_MIN_LR, TAPT_WARMUP), "fill the TAPT_* values"
    # ---- 1. token stream of all Phase-3 text (docs separated by EOS, like your Phase-1 data) ----
    stream = []
    for dom in ("CL", "IT"):
        for split in ("train", "dev", "test"):
            for d in RAW[(dom, split)]:
                for s in d["text"]:
                    stream.extend(tok.encode(" " + s.strip()))
                stream.append(EOS_ID)
    tapt_ids = np.array(stream, dtype=np.int64)
    tok_per_step = TAPT_MICRO_B * TAPT_ACCUM * TAPT_T
    print(f"Phase-3 text: {len(tapt_ids):,} tokens | tokens/step {tok_per_step:,} | "
          f"one pass over the text ≈ {len(tapt_ids) / tok_per_step:.1f} steps | your TAPT_STEPS = {TAPT_STEPS} "
          f"≈ {TAPT_STEPS * tok_per_step / len(tapt_ids):.0f} passes")

    def tapt_batch(B, T):
        s = np.random.randint(0, len(tapt_ids) - T - 1, size=B)
        return torch.from_numpy(np.stack([tapt_ids[i:i + T] for i in s])).to(device)

    # ---- 2. your MLM masking (verbatim from arch3-mlm) ----
    def mlm_mask(x, p=0.15):
        labels = x.clone()
        sel = (torch.rand(x.shape, device=x.device) < p) & (x != EOS_ID)
        labels[~sel] = -100
        r = torch.rand(x.shape, device=x.device); x = x.clone()
        x[sel & (r < 0.8)] = MASK_ID
        rnd = sel & (r >= 0.8) & (r < 0.9)
        x[rnd] = torch.randint(0, EOS_ID, (int(rnd.sum()),), device=x.device)
        return x, labels

    # ---- 3. encoder with its tied LM head ----
    if ARCH in ("arch2", "arch3"):
        lm, lm_cfg = load_encoder(find_input(CKPTS[ARCH]["mlm"]))
    elif ARCH == "arch4":
        lm, lm_cfg = load_encoder_a4(find_input(CKPTS[ARCH]["mlm"]))
    else:
        raise ValueError("TAPT needs a bidirectional (MLM) checkpoint: arch2 / arch3 / arch4")
    lm = lm.to(device).train()
    V = lm_cfg.vocab_size
    def lm_logits(x):                                    # bidirectional over the whole window
        return lm.head(backbone_states(lm, x, torch.full((x.size(0),), x.size(1), device=device)))

    @torch.no_grad()
    def tapt_val(n=8):
        st = np.random.get_state(); np.random.seed(1234); torch.manual_seed(1234); lm.eval(); tot = 0.0
        for _ in range(n):
            xm, lab = mlm_mask(tapt_batch(TAPT_MICRO_B, TAPT_T), TAPT_MASK_P)
            with torch.autocast(DEV_TYPE, dtype=torch.float16, enabled=(DEV_TYPE == "cuda")):
                tot += F.cross_entropy(lm_logits(xm).float().view(-1, V), lab.view(-1), ignore_index=-100).item()
        np.random.set_state(st); lm.train(); return tot / n

    # ---- 4. training loop (same schedule shape as your arch3-mlm: warmup + cosine) ----
    decay = [p for _, p in lm.named_parameters() if p.dim() >= 2]
    nodecay = [p for _, p in lm.named_parameters() if p.dim() < 2]
    opt_t = torch.optim.AdamW([{"params": decay, "weight_decay": 0.1}, {"params": nodecay, "weight_decay": 0.0}], lr=TAPT_PEAK_LR)
    scaler_t = torch.amp.GradScaler(DEV_TYPE, enabled=(DEV_TYPE == "cuda"))
    def tapt_lr(s):
        if s < TAPT_WARMUP: return TAPT_PEAK_LR * (s + 1) / TAPT_WARMUP
        r = (s - TAPT_WARMUP) / max(1, TAPT_STEPS - TAPT_WARMUP)
        return TAPT_MIN_LR + 0.5 * (TAPT_PEAK_LR - TAPT_MIN_LR) * (1 + math.cos(math.pi * min(1.0, r)))
    print("MLM loss on Phase-3 text BEFORE TAPT:", round(tapt_val(), 3))
    t0 = time.time()
    for step in range(TAPT_STEPS):
        for g_ in opt_t.param_groups: g_["lr"] = tapt_lr(step)
        for _ in range(TAPT_ACCUM):
            xm, lab = mlm_mask(tapt_batch(TAPT_MICRO_B, TAPT_T), TAPT_MASK_P)
            with torch.autocast(DEV_TYPE, dtype=torch.float16, enabled=(DEV_TYPE == "cuda")):
                loss = F.cross_entropy(lm_logits(xm).float().view(-1, V), lab.view(-1), ignore_index=-100) / TAPT_ACCUM
            scaler_t.scale(loss).backward()
        scaler_t.unscale_(opt_t); torch.nn.utils.clip_grad_norm_(lm.parameters(), 1.0)
        scaler_t.step(opt_t); scaler_t.update(); opt_t.zero_grad(set_to_none=True)
        if step % 5 == 0 or step == TAPT_STEPS - 1:
            print(f"tapt step {step:4d}/{TAPT_STEPS} | loss {loss.item() * TAPT_ACCUM:.3f} | lr {tapt_lr(step):.2e} | "
                  f"{(time.time() - t0) / 60:.1f} min")
    print("MLM loss on Phase-3 text AFTER TAPT:", round(tapt_val(), 3))

    # ---- 5. save + point this session's fine-tuning at it ----
    tapt_path = f"/kaggle/working/{ARCH}_tapt.pt"
    torch.save({"model": lm.state_dict(), "cfg": asdict(lm_cfg), "step": TAPT_STEPS, "best_val": None}, tapt_path)
    CKPTS[ARCH]["mlm"] = tapt_path
    print("saved", tapt_path, "| CKPTS[ARCH]['mlm'] now points to it (re-running CONTROL resets this)")
    del lm, opt_t, scaler_t; torch.cuda.empty_cache()


### 3.2 Sentence encoding + packing into encoder windows
Every sentence keeps a span `(window, start, end)`; whole sentences are packed into windows of at most `PACK_W` tokens, so a document is encoded with neighbouring sentences as context.

In [ ]:
if PHASE == 3:
    need("MAX_SENT_TOK", "PACK_W", "SENT_SEP_EOS")
    assert HP["MAX_SENT_TOK"] + 1 <= HP["PACK_W"] <= 1024

    SENT_CACHE = {}
    def sent_ids(s):
        if s not in SENT_CACHE:
            SENT_CACHE[s] = tok.encode(" " + s.strip())
        return SENT_CACHE[s]

    def pack_doc(sentences):
        windows, spans, cur, n_trunc, n_empty = [], [], [], 0, 0
        for s in sentences:
            ids = sent_ids(s)
            if not ids: ids, n_empty = [EOS_ID], n_empty + 1
            if len(ids) > HP["MAX_SENT_TOK"]: ids, n_trunc = ids[:HP["MAX_SENT_TOK"]], n_trunc + 1
            piece = ids + ([EOS_ID] if HP["SENT_SEP_EOS"] else [])
            if cur and len(cur) + len(piece) > HP["PACK_W"]:
                windows.append(cur); cur = []
            spans.append((len(windows), len(cur), len(cur) + len(ids)))
            cur = cur + piece
        if cur: windows.append(cur)
        return windows, spans, n_trunc, n_empty

    def build_items(dom, split):
        items, tr, em = [], 0, 0
        for d in RAW[(dom, split)]:
            w, sp, t_, e_ = pack_doc(d["text"]); tr += t_; em += e_
            items.append(dict(id=d["id"], domain=dom, windows=w, spans=sp,
                              labels=[L2I[l] for l in d["labels"]] if "labels" in d else None,
                              sent_idx=list(range(len(sp)))))
        return items, tr, em

    ITEMS = {}
    for dom in ("CL", "IT"):
        for split in ("train", "dev", "test"):
            ITEMS[(dom, split)], tr, em = build_items(dom, split)
            nw = [len(it["windows"]) for it in ITEMS[(dom, split)]]
            print(f"{dom}_{split:5s}: windows/doc p50 {sorted(nw)[len(nw)//2]} max {max(nw)} | "
                  f"truncated sentences {tr} | empty sentences {em}")
    TRAIN = ITEMS[("CL", "train")] + ITEMS[("IT", "train")]
    DEV   = ITEMS[("CL", "dev")]   + ITEMS[("IT", "dev")]
    TEST  = ITEMS[("CL", "test")]  + ITEMS[("IT", "test")]

    # check: spans point at exactly the (possibly truncated) sentence tokens
    it = TRAIN[0]
    ok = all(it["windows"][w][s:e] == sent_ids(it_s)[:e - s] for (w, s, e), it_s in
             zip(it["spans"], RAW[("CL", "train")][0]["text"]) if sent_ids(it_s))
    print("span check on first doc:", ok)

### 3.3 YOUR CODE — the Phase-3 model
**Contract the harness relies on:**
- the Phase-1 backbone is stored as `self.encoder` (so the freezing rule in `apply_technique` works);
- `forward(win_ids, win_lens, spans)` with `win_ids` LongTensor `(n_win, L)` (padded with EOS), `win_lens` LongTensor `(n_win,)`, `spans` = list of `(window, start, end)` per sentence, in document order;
- returns logits `(n_sentences, len(LABELS))`, one row per span, same order;
- get token states with `encoder_states(self.encoder, win_ids, win_lens)` → `(n_win, L, d_model)`.

In [ ]:
import torch
import torch.nn as nn

class RRModel(nn.Module):
    def __init__(self, encoder, d_model, n_labels, dropout=0.1, lstm_hidden=256):
        super().__init__()
        self.encoder = encoder
        self.dropout = nn.Dropout(dropout)
        self.doc_lstm = nn.LSTM(d_model, lstm_hidden, bidirectional=True, batch_first=True)
        self.head = nn.Linear(2 * lstm_hidden, n_labels)
        self.shift_head = nn.Linear(2 * lstm_hidden, 2)

    def forward(self, win_ids, win_lens, spans):
        H = encoder_states(self.encoder, win_ids, win_lens)
        
        S_list = []
        for win_idx, start, end in spans:
            S_list.append(H[win_idx, start:end].mean(dim=0))
        S = torch.stack(S_list)
        
        S = S.unsqueeze(0)
        S, _ = self.doc_lstm(S)
        S = S.squeeze(0)
        
        logits = self.head(self.dropout(S))
        
        return logits

In [ ]:
def build_rr_model():
    enc, d_model = load_backbone()
    m = RRModel(enc, d_model, len(LABELS))
    apply_technique(m)
    return m.to(device)

def build_student():
    need("STUDENT_LAYERS")
    enc, d_model = load_backbone()
    
    enc.blocks = torch.nn.ModuleList(list(enc.blocks)[:HP["STUDENT_LAYERS"]])
    
    m = RRModel(enc, d_model, len(LABELS))
    
    for param in m.parameters():
        param.requires_grad = True
        
    return m.to(device)

def kd_loss(logits, y, cw, teacher=None):
    hard_loss = torch.nn.functional.cross_entropy(logits, y, weight=cw)
    
    if teacher is None:
        return hard_loss
        
    T = HP["KD_T"]
    alpha = HP["KD_ALPHA"]
    
    student_log_probs = torch.nn.functional.log_softmax(logits / T, dim=-1)
    teacher_probs = torch.nn.functional.softmax(teacher / T, dim=-1)
    
    soft_loss = torch.nn.functional.kl_div(
        student_log_probs, 
        teacher_probs, 
        reduction='batchmean'
    ) * (T ** 2)
    
    return (1.0 - alpha) * hard_loss + alpha * soft_loss

### 3.4 Harness (training, evaluation, submission)

In [ ]:
def weighted_ce(logits, y, cw, teacher=None):
    return F.cross_entropy(logits, y, weight=cw)

def crop(item, max_win):
    """Training-time crop to <= max_win consecutive windows (whole sentences only)."""
    n = len(item["windows"])
    if max_win is None or n <= max_win:
        return item
    w0 = random.randint(0, n - max_win)
    keep = [i for i, (w, _, _) in enumerate(item["spans"]) if w0 <= w < w0 + max_win]
    return dict(item, windows=item["windows"][w0:w0 + max_win],
                spans=[(item["spans"][i][0] - w0, item["spans"][i][1], item["spans"][i][2]) for i in keep],
                labels=[item["labels"][i] for i in keep] if item["labels"] is not None else None,
                sent_idx=[item["sent_idx"][i] for i in keep])

def to_tensors(item):
    L = max(len(w) for w in item["windows"])
    x = torch.full((len(item["windows"]), L), EOS_ID, dtype=torch.long)
    for i, w in enumerate(item["windows"]):
        x[i, :len(w)] = torch.tensor(w)
    lens = torch.tensor([len(w) for w in item["windows"]])
    return x.to(device), lens.to(device)

def class_weights_rr(items, power):
    cnt = torch.zeros(len(LABELS))
    for it in items:
        cnt += torch.bincount(torch.tensor(it["labels"]), minlength=len(LABELS)).float()
    w = (cnt.sum() / cnt.clamp(min=1)) ** power
    return (w / w.mean()).to(device)

from sklearn.metrics import f1_score
def macro_f1(gold, pred):
    """Official Phase-3 metric (same as evaluate_phase3.py): macro over ALL 13 labels, zero_division=0."""
    g = [LABELS[x] for d in gold for x in d]
    p = [LABELS[x] for d in pred for x in d]
    per = f1_score(g, p, labels=LABELS, average=None, zero_division=0)
    return float(per.mean()), {l: float(v) for l, v in zip(LABELS, per)}

@torch.no_grad()
def predict_rr(model, items):
    model.eval(); out = {}
    for it in items:
        x, lens = to_tensors(it)
        with torch.autocast(DEV_TYPE, dtype=torch.float16, enabled=(DEV_TYPE == "cuda")):
            lg = model(x, lens, it["spans"])
        out[it["id"]] = lg.float().cpu()
    model.train()
    return out                          # doc id -> (n_sent, n_labels) logits

def evaluate(items, logits_by_id, show_per_class=False):
    res = {}
    for dom in ("CL", "IT", "ALL"):
        sub = [it for it in items if dom == "ALL" or it["domain"] == dom]
        if not sub: continue
        f, per = macro_f1([it["labels"] for it in sub], [logits_by_id[it["id"]].argmax(-1).tolist() for it in sub])
        res[dom] = f
        if show_per_class and dom == "ALL":
            for l, v in per.items(): print(f"    {l:24s} {v:.3f}")
    return res

def trainable_state(model):
    return {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}

def train_rr(model, train_items, dev_items, loss_fn=weighted_ce, teacher=None, *, EPOCHS, LR_ENC, LR_HEAD,
             WARMUP_FRAC, WD, ACCUM_DOCS, CLASS_W_POWER, MAX_TRAIN_WIN, CLIP=1.0, seed=0):
    """Per-document training with gradient accumulation. teacher: doc id -> (n_sent, n_labels) logits (KD runs).
    With dev_items: keeps the best-dev trainable weights. Without: keeps the last epoch."""
    torch.manual_seed(seed); random.seed(seed)
    train_items = list(train_items)
    cw = class_weights_rr(train_items, CLASS_W_POWER)
    enc_p  = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith("encoder.")]
    head_p = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("encoder.")]
    groups = [g for g in ({"params": enc_p, "lr": LR_ENC}, {"params": head_p, "lr": LR_HEAD}) if g["params"]]
    opt = torch.optim.AdamW(groups, weight_decay=WD)
    steps_ep = math.ceil(len(train_items) / ACCUM_DOCS); total = EPOCHS * steps_ep
    warm = max(1, int(WARMUP_FRAC * total))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else max(0.0, (total - s) / max(1, total - warm)))
    scaler = torch.amp.GradScaler(DEV_TYPE, enabled=(DEV_TYPE == "cuda"))
    params = [p for g in groups for p in g["params"]]
    best_f, best_state, hist = -1.0, None, []
    model.train()
    for ep in range(EPOCHS):
        random.shuffle(train_items); t0 = time.time(); tot = 0.0
        for i, item in enumerate(train_items):
            it = crop(item, MAX_TRAIN_WIN)
            x, lens = to_tensors(it)
            y = torch.tensor(it["labels"], device=device)
            tch = teacher[it["id"]][it["sent_idx"]].to(device) if teacher is not None else None
            with torch.autocast(DEV_TYPE, dtype=torch.float16, enabled=(DEV_TYPE == "cuda")):
                lg = model(x, lens, it["spans"])
            loss = loss_fn(lg.float(), y, cw, teacher=tch) / ACCUM_DOCS
            scaler.scale(loss).backward(); tot += loss.item() * ACCUM_DOCS
            if (i + 1) % ACCUM_DOCS == 0 or i + 1 == len(train_items):
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, CLIP)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
        msg = f"epoch {ep + 1}/{EPOCHS} | train loss {tot / len(train_items):.4f} | {time.time() - t0:.0f}s"
        if dev_items:
            res = evaluate(dev_items, predict_rr(model, dev_items))
            msg += " | dev macro-F1 " + " ".join(f"{k} {v:.4f}" for k, v in res.items())
            hist.append(res)
            if res["ALL"] > best_f:
                best_f, best_state = res["ALL"], trainable_state(model)
        print(msg)
    if best_state is not None:
        model.load_state_dict(best_state, strict=False)
        print("BEST dev macro-F1 (ALL):", round(best_f, 4))
    return best_f, hist

def write_submission(logits_by_id, tag):
    pred = {}
    for it in TEST:
        lab = [LABELS[i] for i in logits_by_id[it["id"]].argmax(-1).tolist()]
        assert len(lab) == len(it["spans"]), f"length mismatch for {it['id']}"
        pred[it["id"]] = lab
    want = {d["id"]: len(d["text"]) for dom in ("CL", "IT") for d in RAW[(dom, "test")]}
    assert set(pred) == set(want) and all(len(pred[k]) == want[k] for k in want), "ids / sentence counts do not match the test files"
    assert all(l in L2I for v in pred.values() for l in v)
    jp, zp = f"/kaggle/working/prediction_{tag}.json", f"/kaggle/working/submission_{tag}.zip"
    json.dump(pred, open(jp, "w"))
    with zipfile.ZipFile(zp, "w") as z:
        z.write(jp, arcname="prediction.json")          # must sit at the zip root
    print(f"wrote {zp} | docs {len(pred)} | sentences {sum(map(len, pred.values()))}")

In [ ]:
def focal_loss(logits, y, cw, teacher=None):
    log_pt = torch.nn.functional.log_softmax(logits, dim=-1)
    log_p = log_pt.gather(1, y.unsqueeze(1)).squeeze(1)
    p = torch.exp(log_p)
    focal_factor = (1 - p) ** HP["FOCAL_GAMMA"]
    per_sentence_loss = -cw[y] * focal_factor * log_p
    return per_sentence_loss.mean()

def train_loss():
    if LOSS == "focal":
        need("FOCAL_GAMMA")
        return focal_loss
    return weighted_ce

In [ ]:
import torch

def shift_loss(shift_logits, y):
    targets = torch.zeros_like(y)
    if y.size(0) > 1:
        targets[1:] = (y[1:] != y[:-1]).long()
    return torch.nn.functional.cross_entropy(shift_logits, targets)

### 3.5 Run (uses the CONTROL cell)

In [ ]:
import os
for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
    for n in dirs + files:
        if "arch3" in n or "mlm" in n:
            p = os.path.join(root, n)
            print("DIR " if os.path.isdir(p) else "FILE", p, os.path.getsize(p) if os.path.isfile(p) else "")

In [ ]:
if PHASE == 3:
    need("EPOCHS", "LR_ENC", "LR_HEAD", "WARMUP_FRAC", "WD", "ACCUM_DOCS", "CLASS_W_POWER", "MAX_TRAIN_WIN")
    settings = {k: HP[k] for k in ("EPOCHS", "LR_ENC", "LR_HEAD", "WARMUP_FRAC", "WD", "ACCUM_DOCS",
                                   "CLASS_W_POWER", "MAX_TRAIN_WIN")}
    teacher, loss_fn = None, train_loss()
    if TECHNIQUE == "kd":
        assert TEACHER_TAG, "set TEACHER_TAG in the CONTROL cell and attach that run's teacher-logits file"
        teacher = torch.load(find_input(f"teacher_logits_{TEACHER_TAG}_{RUN_MODE}.pt"))
        model, loss_fn = build_student().to(device), kd_loss
        trainable_report(model, "KD student")
    else:
        model = build_rr_model()

    if RUN_MODE == "dev":
        train_items, dev_items = TRAIN, DEV
    else:
        need("FINAL_EPOCHS"); settings["EPOCHS"] = HP["FINAL_EPOCHS"]
        train_items, dev_items = TRAIN + DEV, None

    t_start = time.time()
    best_f, hist = train_rr(model, train_items, dev_items, loss_fn, teacher, seed=SEED, **settings)
    print(f"training time {(time.time() - t_start) / 60:.1f} min")
    torch.save(trainable_state(model), f"/kaggle/working/weights_{RUN_TAG}.pt")   # trainable params only

    if RUN_MODE == "dev":
        dev_logits = predict_rr(model, DEV)
        print("DEV:", evaluate(DEV, dev_logits, show_per_class=True))
        torch.save(dev_logits, f"/kaggle/working/devlogits_{RUN_TAG}.pt")          # for ensembling later
        test_logits = predict_rr(model, TEST)                                         # submit straight from the best-dev model
        torch.save(test_logits, f"/kaggle/working/testlogits_{RUN_TAG}.pt")
        write_submission(test_logits, RUN_TAG)
        json.dump(hist, open(f"/kaggle/working/hist_{RUN_TAG}.json", "w"))
    else:
        test_logits = predict_rr(model, TEST)
        torch.save(test_logits, f"/kaggle/working/testlogits_{RUN_TAG}.pt")        # for ensembling later
        write_submission(test_logits, RUN_TAG)

    if CACHE_TEACHER_LOGITS:                       # this run will serve as a KD teacher
        tl = predict_rr(model, train_items)
        torch.save(tl, f"/kaggle/working/teacher_logits_{ARCH}_{TECHNIQUE}_{RUN_MODE}.pt")
        print("saved teacher logits for", len(tl), "training docs")

### 3.6 Stage 2 — document layer on full documents
Run right after 3.5 (same session). Writes `submission_<tag>_s2.zip` plus dev/test logits for ensembling.

In [ ]:
# ===== 3.6 Stage 2: train doc_lstm + head on FULL documents (plumbing; your layers, unchanged) =====
# Run right after 3.5 in the same session: it reuses `model`. Encoder + LoRA are frozen; sentence vectors are
# computed once per document (same mean-over-span pooling as RRModel.forward), then only your doc_lstm + head train,
# on whole-document sequences (no MAX_TRAIN_WIN crop).
RUN_STAGE2 = True
if PHASE == 3 and RUN_STAGE2:
    need("STAGE2_EPOCHS", "STAGE2_LR")
    assert all(hasattr(model, a) for a in ("doc_lstm", "head", "dropout")), "needs your BiLSTM RRModel"

    @torch.no_grad()
    def sentence_vectors(m, items, chunk=8):
        m.eval(); out = {}
        for it in items:
            x, lens = to_tensors(it)
            with torch.autocast(DEV_TYPE, dtype=torch.float16, enabled=(DEV_TYPE == "cuda")):
                H = torch.cat([encoder_states(m.encoder, x[i:i + chunk], lens[i:i + chunk])
                               for i in range(0, x.size(0), chunk)])
            out[it["id"]] = torch.stack([H[w, s:e].float().mean(0) for w, s, e in it["spans"]]).cpu()
        return out

    def doc_hidden(m, S):                         # your BiLSTM output per sentence
        out, _ = m.doc_lstm(S.unsqueeze(0).to(device))
        return out.squeeze(0)

    def doc_logits(m, S):                         # = your RRModel.forward after pooling
        return m.head(m.dropout(doc_hidden(m, S)))

    USE_SHIFT = hasattr(model, "shift_head") and HP.get("SHIFT_WEIGHT") is not None
    print("label-shift auxiliary task in Stage 2:", USE_SHIFT)

    def doc_params_of(m):
        ps = list(m.doc_lstm.parameters()) + list(m.head.parameters())
        return ps + (list(m.shift_head.parameters()) if USE_SHIFT else [])

    def stage2_loss(m, S, y, cw, base_loss):
        h = doc_hidden(m, S)
        loss = base_loss(m.head(m.dropout(h)).float(), y, cw)
        if USE_SHIFT:
            loss = loss + HP["SHIFT_WEIGHT"] * shift_loss(m.shift_head(m.dropout(h)).float(), y)
        return loss

    s2_train = list(TRAIN) if RUN_MODE == "dev" else list(TRAIN + DEV)
    SV = sentence_vectors(model, s2_train + DEV + TEST)
    print("sentence vectors cached for", len(SV), "docs")

    for p in model.parameters(): p.requires_grad = False
    doc_params = doc_params_of(model)
    for p in doc_params: p.requires_grad = True
    opt2 = torch.optim.AdamW(doc_params, lr=HP["STAGE2_LR"], weight_decay=HP["WD"])
    cw2 = class_weights_rr(s2_train, HP["CLASS_W_POWER"])

    @torch.no_grad()
    def s2_predict(items):
        model.eval(); out = {it["id"]: doc_logits(model, SV[it["id"]]).float().cpu() for it in items}; model.train()
        return out

    best2, state2 = -1.0, None
    for ep in range(HP["STAGE2_EPOCHS"]):
        random.shuffle(s2_train); model.train(); tot = 0.0
        for it in s2_train:
            loss = stage2_loss(model, SV[it["id"]], torch.tensor(it["labels"], device=device), cw2, train_loss())
            opt2.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(doc_params, 1.0); opt2.step(); tot += loss.item()
        res = evaluate(DEV, s2_predict(DEV))
        print(f"stage2 epoch {ep + 1} | loss {tot / len(s2_train):.4f} | dev " +
              " ".join(f"{k} {v:.4f}" for k, v in res.items()) + ("" if RUN_MODE == "dev" else "  (dev is in training data)"))
        if RUN_MODE == "dev" and res["ALL"] > best2:
            best2, state2 = res["ALL"], {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}
    if state2 is not None:
        model.load_state_dict(state2, strict=False); print("stage2 BEST dev macro-F1:", round(best2, 4))

    print("DEV:", evaluate(DEV, s2_predict(DEV), show_per_class=True))
    tl2 = s2_predict(TEST)
    torch.save(s2_predict(DEV), f"/kaggle/working/devlogits_{RUN_TAG}_s2.pt")
    torch.save(tl2, f"/kaggle/working/testlogits_{RUN_TAG}_s2.pt")
    write_submission(tl2, RUN_TAG + "_s2")


### 3.6b Stage-2 comparison table
List (loss, CLASS_W_POWER) combos; one run prints the official dev score for each and saves their logits for 3.7 / 3.8.

In [ ]:
# ===== 3.6b Stage-2 comparison table (plumbing): one run tries every combo YOU list, prints the official dev score =====
# Needs 3.5 and 3.6 run first in this session (uses their cached sentence vectors). Each combo restarts the doc layer
# from the same post-3.5 weights, so results are comparable. Nothing is chosen automatically: you read the table and pick.
SWEEP = []     # YOUR combos as (loss, CLASS_W_POWER), e.g. [("ce", 0.2), ("focal", 0.2)]
if PHASE == 3 and SWEEP:
    s1 = torch.load(f"/kaggle/working/weights_{RUN_TAG}.pt")
    doc_s1 = {k: v for k, v in s1.items() if k.startswith(("doc_lstm.", "head.", "shift_head."))}
    loss_fns = {"ce": weighted_ce, "focal": focal_loss}
    table = []
    for loss_name, cwp in SWEEP:
        if loss_name == "focal": need("FOCAL_GAMMA")
        model.load_state_dict(doc_s1, strict=False)
        for p in model.parameters(): p.requires_grad = False
        dps = doc_params_of(model)
        for p in dps: p.requires_grad = True
        opt_s = torch.optim.AdamW(dps, lr=HP["STAGE2_LR"], weight_decay=HP["WD"])
        cw_s = class_weights_rr(s2_train, cwp)
        best_s, state_s = -1.0, None
        for ep in range(HP["STAGE2_EPOCHS"]):
            random.shuffle(s2_train); model.train()
            for it in s2_train:
                loss = stage2_loss(model, SV[it["id"]], torch.tensor(it["labels"], device=device), cw_s, loss_fns[loss_name])
                opt_s.zero_grad(set_to_none=True); loss.backward()
                torch.nn.utils.clip_grad_norm_(dps, 1.0); opt_s.step()
            f = evaluate(DEV, s2_predict(DEV))["ALL"]
            if f > best_s:
                best_s, state_s = f, {n: q.detach().cpu().clone() for n, q in model.named_parameters() if q.requires_grad}
        model.load_state_dict(state_s, strict=False)
        tag = f"{RUN_TAG}_s2_{loss_name}_cw{cwp}"
        torch.save(s2_predict(DEV),  f"/kaggle/working/devlogits_{tag}.pt")
        torch.save(s2_predict(TEST), f"/kaggle/working/testlogits_{tag}.pt")
        table.append((loss_name, cwp, best_s))
        print(f"{loss_name:6s} CLASS_W_POWER {cwp}: best dev macro-F1 (official) {best_s:.4f}  -> devlogits/testlogits_{tag}.pt")
    if "state2" in globals() and state2 is not None:      # put 3.6's best doc layer back (for 3.9)
        model.load_state_dict(state2, strict=False)
    print("\nsummary:"); [print(f"  {l:6s} cw {c_}: {f:.4f}") for l, c_, f in table]
    # to submit one: put its testlogits/devlogits file names into 3.7 (single file is fine) or 3.8 (with TAU)


### 3.7 Ensemble
List `testlogits_*.pt` (and optionally `devlogits_*.pt`) files from your runs; writes `submission_p3_ensemble.zip`.

In [ ]:
# ===== 3.7 Ensemble: average probabilities of several runs (plumbing, equal weights) =====
ENS_TEST = []        # e.g. ["testlogits_p3_arch3_lora_dev_s2.pt", "testlogits_p3_arch2_lorma_dev_s2.pt"]
ENS_DEV  = []        # matching devlogits_*.pt files (optional: scores the ensemble on dev)

def _load_logits(name):
    for p in (name, f"/kaggle/working/{name}"):
        if os.path.exists(p): return torch.load(p)
    return torch.load(find_input(name))

def _avg_probs(names):
    acc = None
    for n in names:
        P_ = {k: torch.softmax(v.float(), -1) for k, v in _load_logits(n).items()}
        acc = P_ if acc is None else {k: acc[k] + P_[k] for k in acc}
    return {k: v / len(names) for k, v in acc.items()}

if PHASE == 3 and ENS_DEV:
    print("ENSEMBLE DEV:", evaluate(DEV, _avg_probs(ENS_DEV), show_per_class=True))
if PHASE == 3 and ENS_TEST:
    write_submission(_avg_probs(ENS_TEST), "p3_ensemble")


### 3.8 Logit adjustment (post-processing for rare labels)
Runs on saved logits in seconds; compare TAU values on dev, then set TAU to write a submission.

In [ ]:
# ===== 3.8 Logit adjustment (post-processing; Menon et al. 2021): score_c = log p_c - TAU * log prior_c =====
# Works on saved logits: one file = single model, several files = ensemble average (same as 3.7).
# prior_c = label frequency in TRAIN. TAU = 0 changes nothing; TAU > 0 shifts predictions toward rarer labels.
LA_DEV    = []      # e.g. ["devlogits_p3_arch3_lora_dev_s2.pt"]
LA_TEST   = []      # matching testlogits files, same order
TAU_SWEEP = []      # YOUR candidate values to compare on dev (official metric)
TAU       = None    # YOUR choice after looking at the sweep -> writes the submission

if PHASE == 3 and LA_DEV:
    cnt = torch.zeros(len(LABELS))
    for it in TRAIN:
        cnt += torch.bincount(torch.tensor(it["labels"]), minlength=len(LABELS)).float()
    log_prior = torch.log((cnt + 1) / (cnt + 1).sum())

    def adjust(probs_by_id, tau):
        return {k: torch.log(v.clamp_min(1e-12)) - tau * log_prior for k, v in probs_by_id.items()}

    dev_p = _avg_probs(LA_DEV)
    print("no adjustment:", evaluate(DEV, adjust(dev_p, 0.0)))
    for tau in TAU_SWEEP:
        print(f"tau {tau}: dev", evaluate(DEV, adjust(dev_p, tau)))
    if TAU is not None and LA_TEST:
        print(f"chosen tau {TAU}: dev", evaluate(DEV, adjust(dev_p, TAU), show_per_class=True))
        write_submission(adjust(_avg_probs(LA_TEST), TAU), f"p3_la_tau{TAU}")


### 3.9 kNN interpolation (inference only)
Mixes your model's probabilities with a vote of the k most similar training sentences; prints dev per (k, LAMBDA) you list.

In [ ]:
# ===== 3.9 kNN interpolation at inference (Santosh et al., LREC-COLING 2024) — plumbing, no retraining =====
# Run after 3.6 in the same session. Datastore = your BiLSTM output for every TRAINING sentence + its gold label.
# Each dev/test sentence: its k nearest training sentences (L2 distance) vote with weight exp(-distance) -> p_knn.
# Final = LAMBDA * p_model + (1 - LAMBDA) * p_knn   (LAMBDA = 1 -> your model alone).
# The paper grid-searched LAMBDA in [0, 1] (steps of 0.1) and k over powers of 2 from 8 to 256, on dev.
KNN_K_LIST      = []            # YOUR candidates for k
KNN_LAMBDA_LIST = []            # YOUR candidates for LAMBDA
KNN_K, KNN_LAMBDA = None, None  # YOUR choice after reading the table -> writes the submission + logits files

if PHASE == 3 and KNN_K_LIST and KNN_LAMBDA_LIST:
    if RUN_MODE != "dev":
        print("NOTE: in final mode dev is in the datastore, so dev scores below are not meaningful")

    @torch.no_grad()
    def _reps(items):
        model.eval(); out = {it["id"]: doc_hidden(model, SV[it["id"]]).float() for it in items}; model.train()
        return out

    _tr = _reps(s2_train)
    knn_keys = torch.cat([_tr[it["id"]] for it in s2_train])
    knn_vals = torch.cat([torch.tensor(it["labels"]) for it in s2_train]).to(device)
    q_dev, q_test = _reps(DEV), _reps(TEST)
    pm_dev  = {i: torch.softmax(v, -1) for i, v in s2_predict(DEV).items()}
    pm_test = {i: torch.softmax(v, -1) for i, v in s2_predict(TEST).items()}
    print(f"datastore: {knn_keys.size(0):,} training sentences x {knn_keys.size(1)} dims")

    @torch.no_grad()
    def p_knn(q, k):
        d = torch.cdist(q, knn_keys)                                   # (n, N) L2 distances
        dk, ik = d.topk(k, largest=False)
        w = torch.exp(-(dk - dk[:, :1]))                              # = exp(-d) up to a per-row constant
        P = torch.zeros(q.size(0), len(LABELS), device=q.device)
        P.scatter_add_(1, knn_vals[ik], w)
        return (P / P.sum(-1, keepdim=True)).cpu()

    def knn_mix(qs, pm, k, lam):
        return {i: torch.log(lam * pm[i] + (1 - lam) * p_knn(qs[i], k) + 1e-12) for i in qs}

    print("model alone (LAMBDA = 1):", evaluate(DEV, {i: torch.log(p) for i, p in pm_dev.items()}))
    for k in KNN_K_LIST:
        for lam in KNN_LAMBDA_LIST:
            print(f"k {k:4d} | LAMBDA {lam:.2f} | dev", evaluate(DEV, knn_mix(q_dev, pm_dev, k, lam)))

    if KNN_K is not None and KNN_LAMBDA is not None:
        dl = knn_mix(q_dev, pm_dev, KNN_K, KNN_LAMBDA)
        tl = knn_mix(q_test, pm_test, KNN_K, KNN_LAMBDA)
        print("chosen:", evaluate(DEV, dl, show_per_class=True))
        tag = f"{RUN_TAG}_knn_k{KNN_K}_l{KNN_LAMBDA}"
        torch.save(dl, f"/kaggle/working/devlogits_{tag}.pt")       # log-probs: usable in 3.7 / 3.8 like any logits
        torch.save(tl, f"/kaggle/working/testlogits_{tag}.pt")
        write_submission(tl, tag)


# PHASE 4 — Court Judgment Prediction
_Section added when the phase opens. Data files are already listed in `PHASE_FILES`._

# PHASE 5 — Legal Statute Identification
_Section added when the phase opens. Data files are already listed in `PHASE_FILES`._

# PHASE 6 — Abstractive Summarization (uses the causal 'lm' checkpoints)
_Section added when the phase opens. Data files are already listed in `PHASE_FILES`._

# PHASE 7 — Prior Case Retrieval
_Section added when the phase opens. Data files are already listed in `PHASE_FILES`._